This notebook generates all figures for the qualitative study.

- Figures are first created in HTML to support markdown, latex, code and svg in SFT prompts.
- PDFs are generated with `playwright`.
- See `./figures/case_study/main.tex`

In [ ]:

BASE_MODEL =   "Olmo-3-1025-7B"
# BASE_MODEL = "Qwen3-0.6B-Base" # for debugging


MODELS = [
    BASE_MODEL + sft for sft in [
    "_squad_v2_conversational_lr0.0001_seed42",
    "_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42",
    "_ftrace_date-lm_conversational_lr0.0001_seed42",
    "_tulu-3-sft-personas-code_lr0.0001_seed42",
    "_tulu-3-sft-personas-math_lr0.0001_seed42"]
]

SCRIPTS = [
    "representations/run_gradsim_unnormalized.py",
    "representations/run_bm25.py",
    "representations/run_embd_llama-embed-nemotron-8b.py",
    "representations/run_repsim.py"
]

In [2]:
from load_results import abbrev_map
import glob
import subprocess
from structural_similarity import load_results, build_inner_args
import yaml
import glob
import subprocess
import os
from argparse import Namespace
from color import darken_color, base_colors, saturate_color, vivid_color

from collections import defaultdict 
import torch
import numpy as np
from datasets import load_dataset
import matplotlib.colors as mcolors

import numpy as np
import matplotlib.patches as patches
from matplotlib.image import BboxImage
from matplotlib.transforms import Bbox


from IPython.display import display, HTML
import html
import re
import markdown

import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

In [3]:




with open("experiments.yaml", "r") as f:
    cfg = yaml.safe_load(f)

base_models = [b.split('/')[1] for b in cfg["base_models"]]
experiments_ = cfg["experiments"]

experiments = []
for base in base_models:
    for exp in experiments_:
        new_exp = exp.copy()
        new_exp["model"] = f"./models/{base}{exp['model']}"
        if not os.path.exists(new_exp["model"]):
            print(f"[ERROR] Model {new_exp['model']} not found")
            exit(-1)
        experiments.append(new_exp)



data = defaultdict(list)
for script_path in SCRIPTS:
    for exp in experiments:
        if not any([model in exp["model"] for model in MODELS]):
            continue

        args_dict = {
            "model": exp["model"],
            "train_dataset_name": exp["dataset"],
            "train_dataset_split": "test",
            "test_dataset_name": exp["dataset"],
            "test_dataset_split": "test",
        }

        args = Namespace(**args_dict)
        module_name = script_path.replace(".py", "").replace("/", ".")
        inner_args = build_inner_args(args, func_only=False)
        representation_builder = lambda module_name=module_name, inner_args=inner_args: load_results(
            module_name,
            inner_args
        )[0]
        data[exp["model"]].append((module_name, args,representation_builder))   


In [4]:
EPS = 1e-8

def standardize(x):
    x = np.array(x)
    return (x - x.mean()) / (x.std() + EPS)

def get_doc(dataset, idx):
    if idx is None:
        return None
    ex = dataset[idx]
    return ex["messages"]


def analyze_representations(representations, dataset, k=10, metadata_only=False):
    reps = {name: tensor.float() if not callable(tensor) else tensor for name, _, tensor in representations}
    names = list(reps.keys())
    knn_maps = {}
    results = {}
    if dataset is not None:
        n = len(dataset)

    for i in range(len(names)):
        for j in range(i + 1, len(names)):

            r_i, r_j = names[i], names[j]
            
            if metadata_only:
                results[(r_i, r_j)] = {
                }
                continue
            X_i = reps[r_i]
            X_j = reps[r_j]
            
            X_i = X_i - X_i.mean(dim=0, keepdim=True)
            X_j = X_j - X_j.mean(dim=0, keepdim=True)

            S_i = X_i @ X_i.T   
            S_j = X_j @ X_j.T   


            idx = torch.arange(n)
            S_i[idx, idx] = -1e9
            S_j[idx, idx] = -1e9

            knn_i = torch.topk(S_i, k=k, dim=1).indices  # (n, k)
            knn_j = torch.topk(S_j, k=k, dim=1).indices
            knn_maps[r_i] = knn_i
            knn_maps[r_j] = knn_j

            overlaps = []
            knn_i_np = knn_i.cpu().numpy()
            knn_j_np = knn_j.cpu().numpy()

            for q in range(n):
                overlaps.append(
                    len(set(knn_i_np[q]) & set(knn_j_np[q])) / k
                )

            overlaps = np.array(overlaps)
            
            # all indices tied for max/min
            max_candidates = np.flatnonzero(overlaps == overlaps.max())
            min_candidates = np.flatnonzero(overlaps == overlaps.min())

            # randomly choose one
            np.random.seed(42)
            max_idx = int(np.random.choice(max_candidates))
            min_idx = int(np.random.choice(min_candidates))

            results[(r_i, r_j)] = {
                "overlaps": overlaps,
                "max_overlap_example": max_idx,
                "min_overlap_example": min_idx,
     
            }

    return results, knn_maps

In [5]:
def align_knn(idx_a, idx_b):
    """
    this helper function aligns overlapping items for plotting to avoid having redundant examples across rows
    """
    used = set()
    b_set = set(idx_b)

    aligned = []

    # match in order of idx_a
    for i in idx_a:
        if i in b_set:
            aligned.append(i)
            used.add(i)
        else:
            aligned.append(None)

    # remaining elements from idx_b not used
    backfill = iter([x for x in idx_b if x not in used])

    return [
        v if v is not None else next(backfill, None)
        for v in aligned
    ]




In [6]:
def preprocess_latex(text):
    # \[ ... \] → $$ ... $$
    text = re.sub(r"\\\[\s*", r"$$", text)
    text = re.sub(r"\s*\\\]", r"$$", text)

    # \( ... \) → $ ... $
    text = re.sub(r"\\\((.*?)\\\)", r"$\1$", text)

    return text

def render_message(msg):
    role = msg["role"]
    content = preprocess_latex(msg["content"])

    content_html = markdown.markdown(
        content,
        extensions=["fenced_code", "tables"]
    )

    side = "left" if role == "assistant" else "right"
    bubble_class = "assistant" if role == "assistant" else "user"

    return f"""
    <div class="msg-row {side}">
        <div class="bubble {bubble_class}">
            {content_html}
        </div>
    </div>
    """


def render_chat(messages, top=False):
    if top:
        return "<div class='mini-chat' id='top'>" + "".join(
        render_message(m) for m in messages[:2]
    ) + "</div>"
    else:
        return "<div class='mini-chat'>" + "".join(
            render_message(m) for m in messages[:2]
        ) + "</div>"



In [7]:
def plot_chat_case_study_html(
    example_to_explain,
    docs,
    html_path=None,
    onecol=False,
    color=None,
    color_top=None,
    C=None,
    max_chat_height=6*6,
):

    if C is None:
        C = 3 if onecol else 5

    docs = [
        (method_name, docs_col)
        for method_name, docs_col in docs
        if method_name != "FL λ=0"
    ]

    top_chat_html = render_chat(example_to_explain, top=True)

    grid_rows_html = ""
    merged_positions = set()


    row_labels = []
    for r, (method_name, docs_col) in enumerate(docs):

        row_html = ""

        label_cell = f"""
        <td class="row-label" colspan="{C}">
            {method_name.replace("Top-5", "").replace("FL", "")}
        </td>
        """
        row_labels.append(label_cell)

       
        for c, doc in enumerate(docs_col[:C]):

            if (r, c) in merged_positions:
                continue

            if doc is None:
                continue

            rowspan = 1

            should_merge = (
                r + 1 < len(docs)
                and c < len(docs[r + 1][1])
                and docs[r + 1][1][c] is None
            )

            if should_merge:
                rowspan = 2
                merged_positions.add((r + 1, c))

            row_html += f"""
            <td class="cell" rowspan="{rowspan}">
                {render_chat(doc)}
            </td>
            """

        grid_rows_html += f"<tr>{row_html}</tr>"


    html_output = f"""
    <style>
    .container {{
        display: flex;
        font-size: 6pt;
        width: 455.24411pt;
        height: 300pt;
        background-color: white;
        margin: auto;
        font-family: -apple-system, BlinkMacSystemFont, sans-serif;
    }}

    body {{
        margin: 0;
    }}
    .container * {{
        font-size: inherit;
    }}

    /* ---------------- TABLE ---------------- */
    .layout-table {{
    width: 100%;
    border-collapse: collapse;
    table-layout: fixed;

    flex: 1;
    
    grid-template-rows: auto auto 1fr; /* adjust if needed */
    }}
    .row-label{{
        font-size: 8pt;
        text-align: center;
    }}
    /* ---------------- TOP CHAT ---------------- */
    .top-chat-cell {{
        padding: 0;
        padding-top: 1px;
        padding-bottom: 2px;
        background: white;
    }}
    tr:nth-of-type(2) .row-label::after {{
        content: "↓";
        display: block;
        margin-top: 0;
        font-size: 7pt;
        line-height: 1;
    }}
    tr:nth-of-type(5) .row-label::before {{
    content: "↑";
    display: block;
    margin-top: 0;
    font-size: 7pt;
    line-height: 1;
    }}
    .top-chat {{
        width: 50%;
        margin: 0 auto 25px auto;
        padding: 16px;
    
        border-radius: 14px;
    }}
    .layout-table tr:nth-child(4) td {{
        border-top: 1px solid #ddd;
    }}
    /* ---------------- ROW LABEL ---------------- */
    .row-label{{
        font-size: 8pt;
        text-align: center;
    }}

    /* ---------------- CELL ---------------- */
    .cell {{
      
        padding: 0;
 
        vertical-align: middle;
        padding-top: 4pt;
        padding-bottom: 4pt;
     
        overflow: hidden;
        padding-left: 4pt;
    }}

    /* ---------------- BUBBLES ---------------- */
    .bubble {{
        padding: 4px;
        border-radius: 4px;
        max-width: 90%;
        max-height: 35pt;
        position: relative;
        display: inline-block;
    }}
    .msg-row.left{{
        padding-top: 1pt;
    }}
    .assistant {{
        background: #E5E5EA;
        color: black;
        margin-right: 10%;
    }}

    .user {{
        background: {color};
        color: white;
        margin-left: 10%;
    }}

    #top .bubble.user {{
        background: {color_top};
    }}

    /* ---------------- TEXT-ONLY FADE (FINAL FIX) ---------------- */
    .bubble.is-overflowing {{
        -webkit-mask-image: linear-gradient(
            to bottom,
            rgba(0,0,0,1) 0%,
            rgba(0,0,0,1) 75%,
            rgba(0,0,0,0) 100%
        );

        mask-image: linear-gradient(
            to bottom,
            rgba(0,0,0,1) 0%,
            rgba(0,0,0,1) 75%,
            rgba(0,0,0,0) 100%
        );
        border-bottom-left-radius: 0;
        border-bottom-right-radius: 0;
    }}

    /* ---------------- CODE ---------------- */
    pre {{
        padding: 8px;
        border-radius: 10px;
        overflow-x: auto;
    }}

    code {{
        font-family: Menlo, Monaco, monospace;
    }}

    p {{
        margin: 0;
        padding: 0;
    }}

    ul, ol {{
        margin: 0;
        padding: 0;
        list-style-position: inside;
    }}

    li {{
        margin: 0;
        padding: 0;
        line-height: 1.15;
    }}
    #top {{
        max-width: 50%;
        margin: auto;
    }}


    li p {{
        margin: 0;
        padding: 0;
    }}
    </style>

    <!-- MathJax -->
    <script>
    window.MathJax = {{
      tex: {{
        inlineMath: [['$', '$'], ['\\\\(', '\\\\)']]
      }},
      startup: {{
        ready() {{
          MathJax._.core.MmlTree.MmlNodes.math.MmlMath.defaults.scriptminsize = '4pt';
          MathJax.startup.defaultReady();
        }}
      }}
    }};
    </script>

    <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script>

    <script>
    window.onload = function () {{

        function applyOverflow() {{
            document.querySelectorAll('.bubble').forEach(el => {{
                requestAnimationFrame(() => {{
                    if (el.scrollHeight > el.clientHeight) {{
                        el.classList.add('is-overflowing');
                    }}
                }});
            }});
        }}

        if (window.MathJax) {{
            MathJax.typesetPromise().then(applyOverflow);
        }} else {{
            applyOverflow();
        }}
    }};
    </script>

    <div class="container">

        <table class="layout-table">
        <colgroup>
            <col style="width: calc((100%) / {C});" span="{C}">
        </colgroup>

        <tr>
            <td colspan="{C}" class="top-chat-cell">
                {top_chat_html}
            </td>
        </tr>
        
        <tr>
        {row_labels[0]}
        </tr>
        
        {grid_rows_html}
        <tr>
        {row_labels[1]}
        </tr>
        </table>

    </div>
    """

    if html_path:
        html_path = html_path.replace(".pdf", ".html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(html_output)

        print(f"Saved HTML to {html_path}")

In [8]:
import shutil

for model_name, representation_data in data.items():
    # shutil.rmtree("/tmp/cache/representations", ignore_errors=True)

    
    representations = [(r, args_,  d_fn()) for r, args_, d_fn in representation_data]
    
    namespace = representations[0][1]
    dataset_name = namespace.train_dataset_name
    dataset_split = namespace.train_dataset_split


    dataset = load_dataset(dataset_name, split=dataset_split)
    results,knn_maps = analyze_representations(representations, dataset, k=5)

    for pair, stats in results.items():
        r_i, r_j = pair
        filename = lambda mode: f"./figures/case_study/{os.path.basename(model_name).replace('_','-')}/{r_i.split('.')[-1]}_{os.path.basename(r_j.split('.')[-1])}_{mode}.html"      
        max_file = filename("max")
        min_file = filename("min") 
      

    
        def process(mode, idx_key):
            print(mode)
            idx = stats[idx_key]

            idx_a = knn_maps[r_i][idx].tolist()
            idx_b = knn_maps[r_j][idx].tolist()

            aligned_b = align_knn(idx_a, idx_b)

            idx_a_sorted, idx_b_sorted = zip(*[(g[0],(g[1] if (g[0] != g[1]) else None)) for g in
                sorted(
                    zip(idx_a, aligned_b),
                    key=lambda x: x[0] == x[1],
                    reverse=True
                )
            ])
            

            return idx, idx_a_sorted, idx_b_sorted

        
        # ---------- MAX ----------
        max_idx, idx_a_max_sorted, idx_b_max_sorted = process("max", "max_overlap_example")
        os.makedirs(os.path.dirname(max_file), exist_ok=True)
        plot_chat_case_study_html(
            get_doc(dataset, max_idx),
            [
                (abbrev_map[pair[0]], [get_doc(dataset, i) for i in idx_a_max_sorted]),
                (abbrev_map[pair[1]], [get_doc(dataset, i) for i in idx_b_max_sorted]),
            ],
            html_path=max_file,
            # onecol=False,
            color_top=mcolors.to_hex(base_colors[0]),
            color=mcolors.to_hex(darken_color(base_colors[1], 1))
        )


        # ---------- MIN ----------
        min_idx, idx_a_min_sorted, idx_b_min_sorted = process("min", "min_overlap_example")

        plot_chat_case_study_html(
            get_doc(dataset, min_idx),
            [
                (abbrev_map[pair[0]], [get_doc(dataset, i) for i in idx_a_min_sorted]),
                (abbrev_map[pair[1]], [get_doc(dataset, i) for i in idx_b_min_sorted]),
            ],
            html_path=min_file,
            color_top=mcolors.to_hex(base_colors[0]),
            color=mcolors.to_hex(darken_color(base_colors[2], 1))
        )

In [9]:
from pathlib import Path
import asyncio
from playwright.async_api import async_playwright
import nest_asyncio

nest_asyncio.apply()

ROOT_DIR = Path(r"figures/case_study")


async def render_html_to_pdf(page, html_path: Path):
    url = html_path.resolve().as_uri()
    pdf_path = html_path.with_suffix(".pdf")

    await page.goto(url, wait_until="networkidle")

    await page.add_style_tag(content="""
        @page { margin: 0; }
        body { margin: 0; }
    """)

    box = await page.eval_on_selector(
        ".container",
        """el => {
            const r = el.getBoundingClientRect();
            return { width: r.width, height: r.height };
        }"""
        )
    await page.add_style_tag(content="""
        * {
            -webkit-print-color-adjust: exact !important;
            print-color-adjust: exact !important;
        }

        /* FORCE layer flattening */
        * {
            opacity: 1 !important;
            backdrop-filter: none !important;
            filter: none !important;
            transform: none !important;
        }
    """)
    await page.pdf(
        path=str(pdf_path),
        print_background=True,
        width=f"{box['width']}px",
        height=f"{box['height']*1.05}px",
        margin={
            "top": "0px",
            "bottom": "0px",
            "left": "0px",
            "right": "0px"
        }
    )

    print(f"Saved: ./{pdf_path}")


async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()

        html_files = list(ROOT_DIR.rglob("*.html"))

        for html_file in html_files:
            try:
                await render_html_to_pdf(page, html_file)
            except Exception as e:
                print(f"Failed: {html_file} -> {e}")

        await browser.close()



await main()

In [10]:
import os

output_path = "figures/case_study/main.tex"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w") as f:  # use "a" if you want to append instead
    for model_name, representation_data in data.items():
        representations = [(r, args_, d_fn) 
                           for r, args_, d_fn in representation_data]
      
        namespace = representations[0][1]
        dataset_name = namespace.train_dataset_name
        dataset_split = namespace.train_dataset_split

        results, _ = analyze_representations(representations, None, k=5, metadata_only=True)

        for pair, stats in results.items():
            r_i, r_j = pair

            filename = lambda mode: (
                f"./figures/case_study/"
                f"{os.path.basename(model_name).replace('_','-')}/"
                f"{r_i.split('.')[-1]}_{os.path.basename(r_j.split('.')[-1])}_{mode}.pdf"
            )

            max_file = filename("max")
            min_file = filename("min")
            model_clean = os.path.basename(model_name).replace('Olmo-3-1025-7B', 'Olmo-3-7B') \
                                    .replace('_lr0.0001_seed42', '') \
                                    .replace('_', '\\_')
            latex = f"""
\\begin{{figure*}}
    \\centering

    \\begin{{minipage}}{{1\\textwidth}}
        \\centering
        \\includegraphics[width=\\linewidth]{{{max_file}}}
    \\end{{minipage}}

    \\begin{{minipage}}{{1\\textwidth}}
        \\centering
        \\includegraphics[width=\\linewidth]{{{min_file}}}
        \\caption{{Instances with highest (top plot) and lowest (bottom plot) 5-nearest neighbor overlap between {abbrev_map[r_i]} and {abbrev_map[r_j]} for \\texttt{{{model_clean}}}.}}
    \\end{{minipage}}

\\end{{figure*}}

"""

            f.write(latex + "\n\n")

      
            continue